<a href="https://colab.research.google.com/github/paulocesardev8/descomplica-faculdade/blob/main/integra%C3%A7%C3%A3o_com_chatbots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install PyQt5


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 282.4/282.4 kB 22.9 MB/s eta 0:00:00


In [1]:
!pip install nltk

In [ ]:
import sys
import nltk
from PyQt5.QtWidgets import QApplication, QMainWindow, QTextEdit, QLineEdit, QPushButton, QVBoxLayout, QWidget
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Baixar os recursos necessários do NLTK
nltk.download('punkt')
nltk.download('stopwords')


# Definição do modelo de dados
class Pergunta:
    def __init__(self, id, texto):
        self.id = id
        self.texto = texto


class Resposta:
    def __init__(self, id, pergunta_id, texto):
        self.id = id
        self.pergunta_id = pergunta_id
        self.texto = texto


# Base de dados de perguntas e respostas
perguntas = [
    Pergunta(1, "Qual é a capital da França?"),
    Pergunta(2, "Como posso aprender Python?"),
    Pergunta(3, "O que é inteligência artificial?")
]

respostas = [
    Resposta(1, 1, "A capital da França é Paris."),
    Resposta(2, 2, "Você pode aprender Python através de cursos online."),
    Resposta(3, 3, "Inteligência artificial é um campo da ciência da computação.")
]


# Função para pré-processar o texto
def preprocessar_texto(texto):
    tokens = word_tokenize(texto.lower())
    stop_words = set(stopwords.words('portuguese'))
    tokens_filtrados = [token for token in tokens if token not in stop_words and token.isalpha()]
    return ' '.join(tokens_filtrados)


# Função para encontrar a melhor correspondência
def encontrar_resposta(pergunta_usuario):
    # Pré-processar as perguntas e respostas
    perguntas_processadas = [preprocessar_texto(p.texto) for p in perguntas]

    # Adicionar a pergunta do usuário à lista de perguntas
    perguntas_processadas.append(preprocessar_texto(pergunta_usuario))

    # Calcular a similaridade usando TF-IDF
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(perguntas_processadas)

    # Calcular similaridade entre a pergunta do usuário e as perguntas no banco de dados
    similaridade = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1])

    # Encontrar o índice da pergunta mais semelhante
    indice_mais_semelhante = similaridade.argmax()

    # Retornar a resposta correspondente
    resposta_mais_semelhante = respostas[indice_mais_semelhante]
    return resposta_mais_semelhante.texto


# Definição da interface gráfica usando PyQt5
class ChatbotApp(QMainWindow):
    def __init__(self):
        super().__init__()

        self.setWindowTitle('Chatbot')
        self.setGeometry(100, 100, 500, 400)

        # Layout da interface
        layout = QVBoxLayout()

        # Caixa de texto para exibir o chat
        self.chat_box = QTextEdit(self)
        self.chat_box.setReadOnly(True)
        layout.addWidget(self.chat_box)

        # Entrada de texto para a pergunta do usuário
        self.input_box = QLineEdit(self)
        layout.addWidget(self.input_box)

        # Botão de enviar
        self.send_button = QPushButton('Enviar', self)
        self.send_button.clicked.connect(self.enviar_pergunta)
        layout.addWidget(self.send_button)

        # Widget central para o layout
        container = QWidget()
        container.setLayout(layout)
        self.setCentralWidget(container)

    # Função para enviar a pergunta
    def enviar_pergunta(self):
        pergunta_usuario = self.input_box.text()
        if pergunta_usuario.lower() == 'sair':
            self.close()
        else:
            resposta = encontrar_resposta(pergunta_usuario)
            self.chat_box.append("Você: " + pergunta_usuario)
            self.chat_box.append("Chatbot: " + resposta)
            self.input_box.clear()


# Função principal para rodar o aplicativo
def main():
    app = QApplication(sys.argv)
    chatbot = ChatbotApp()
    chatbot.show()
    sys.exit(app.exec_())


if __name__ == "__main__":
    main()

### Testando a função `encontrar_resposta` diretamente

Como a interface gráfica não é visível no ambiente Colab, podemos testar a funcionalidade do chatbot chamando a função `encontrar_resposta` diretamente. Isso demonstrará que a lógica de processamento de perguntas e respostas está funcionando corretamente.